In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
from urllib.parse import urljoin

# Day 5: Automated Web Scraping & AI Brochure Generation

This notebook demonstrates a **practical LLM application**: automatically generate company brochures by:

1. **Web Scraping**: Extract links and content from a company website
2. **Link Filtering**: Use AI to identify relevant pages (About, Careers, Pricing, etc.)
3. **Content Aggregation**: Fetch full content from selected pages
4. **AI Brochure Generation**: Use LLM to synthesize a professional brochure
5. **Streaming**: Display responses in real-time as they're generated

**Key concepts:**
- System prompts for tone control
- JSON-structured responses from LLMs
- Streaming API responses
- Prompt engineering with examples
- Real-world end-to-end automation

Let's build an automated brochure generator!

## Setup

Configure Ollama as the local LLM provider using the OpenAI-compatible API.

```python
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
```

In [ ]:
links = fetch_website_links("https://edwarddonner.com")
links

## Step 1: Extract All Links from a Website

First, we scrape all links from a target URL using the custom `scraper` module.

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

## Step 2: Design a System Prompt for Link Filtering

**Problem**: Not all links are useful for a brochure. We fetch hundreds of links but only need important ones (About, Careers, etc.).

**Solution**: Use an LLM with a system prompt + JSON response format to intelligently filter links.

**Prompt Engineering Notes:**
- Define the task clearly
- Specify output format (JSON in this case)
- Give examples for expected structure
- Tell the model what to exclude (privacy, terms, email links)

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    links = [urljoin(url, link) for link in links]
    user_prompt += "\n".join(links)
    return user_prompt

## Step 3: Build the User Prompt with Actual Links

This function:
1. Fetches all links from the target URL
2. Converts relative links to absolute URLs using `urljoin()`
3. Constructs a user prompt with the full link list
4. The LLM will then evaluate which links are relevant

**Key detail**: We need full URLs (`https://...`) for the LLM to understand what each link is.

In [ ]:
print(get_links_user_prompt("https://edwarddonner.com"))

In [ ]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

## Step 4: Call the LLM to Select Relevant Links

**Key feature**: `response_format={"type": "json_object"}`

This tells the LLM to respond **only in valid JSON format**, making it easy to parse the result with `json.loads()`. This is called **constrained output** or **structured generation**.

The function:
1. Sends the system prompt (link filtering instructions) + user prompt (actual links)
2. Specifies JSON output format
3. Parses the JSON response into Python
4. Returns a structured dict of relevant links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling llama3.2")
    response = openai.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

## Step 5: Aggregate Content from All Relevant Pages

This function orchestrates the full pipeline:

1. **Fetch landing page content** - The main website text
2. **Select relevant links** - Use AI to pick important pages
3. **Fetch each relevant link's content** - Get full text from About, Careers, etc.
4. **Combine everything** - Return markdown with all content organized by link type

This creates a comprehensive knowledge base of the company that the LLM can use to generate a brochure.

In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


## Step 6: Design System Prompt for Brochure Generation

**Power of Prompting**: Notice the commented-out "humorous version" below the standard prompt.

This demonstrates **tone engineering**—the exact same content pipeline can produce:
- Professional brochures (standard prompt)
- Humorous/entertaining brochures (just change the system prompt)
- Marketing-focused brochures (tweak the prompt again)

**Prompt elements:**
- Define the task clearly
- Specify output format (markdown)
- List what to include (culture, customers, careers)
- Set tone/style (professional, humorous, etc.)

In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

## Step 7: Build User Prompt with Aggregated Content

**Design consideration**: **Token limits & cost control**

The `[:5_000]` truncation is crucial because:
- LLMs have max token limits (context windows)
- Longer inputs = slower processing & higher costs
- We keep the most useful parts and trim excess

This is a practical engineering decision: use enough content for quality output, but not so much that it's wasteful.

In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

## Step 8: Generate and Display the Brochure

This function:
1. Sends the system prompt + aggregated company content to the LLM
2. Gets back a markdown-formatted brochure
3. Uses IPython's `display(Markdown(...))` to render it nicely in the notebook

**Result**: A professional brochure automatically generated from website content!

In [ ]:
create_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

## Step 9: Streaming for Real-Time Updates

**What is streaming?**

Instead of waiting for the complete response, streaming returns results **piece by piece** as they're generated.

**Benefits:**
- **Faster perceived response** - You see content appearing in real-time
- **Better UX** - No long wait times before output appears
- **Lower latency** - Start displaying before full generation is done

**How it works here:**
1. `stream=True` enables streaming mode
2. Each `chunk` contains a partial response
3. We accumulate chunks into `response` and update the display live
4. `update_display()` refreshes the notebook cell content on each iteration

Try this for a long response—you'll see text appear word-by-word!

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """
#
# stream_brochure("HuggingFace", "https://huggingface.co")

## Key Takeaways

✅ **Multi-step LLM pipelines** - Link filtering, content aggregation, brochure generation chained together  
✅ **Structured output** - JSON responses make results machine-readable  
✅ **Prompt engineering** - Tweak system prompts to change tone, style, focus  
✅ **Streaming** - Real-time responses for better UX  
✅ **Real-world application** - Fully automated company brochure generator  

**Try experimenting with:**
- Different websites  
- The humorous brochure prompt version  
- Different page selection criteria  
- Adding fact-checking or validation steps  
- Saving the brochure to a file  

This demonstrates how to build production-ready LLM applications with web data!